In [1]:
import functools
import jax
import os

from datetime import datetime
from jax import numpy as jp
import matplotlib.pyplot as plt

from IPython.display import HTML, clear_output

import brax
import flax
from brax import envs
from brax.io import model
from brax.io import json
from brax.io import html
# from brax.training.agents.ppo import train as ppo
# from brax.training.agents.sac import train as sac

import functools
import time
from typing import Any, Callable, Mapping, Optional, Tuple, Union

from absl import logging
from brax import base
from brax import envs
from brax.training import acting
from brax.training import gradients
from brax.training import pmap
from brax.training import types
from brax.training.acme import running_statistics
from brax.training.acme import specs
from brax.training.agents.ppo import losses as ppo_losses
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.types import Params
from brax.training.types import PRNGKey
from brax.v1 import envs as envs_v1
from etils import epath
import flax
import jax
import jax.numpy as jnp
import numpy as np
import optax
from orbax import checkpoint as ocp

from brax.envs.base import PipelineEnv, State
from brax.io import mjcf
from etils import epath

import wandb
import xmltodict
from time import strftime, gmtime

from ppo import ppo_train
from env import HalfcheetahWithObstacles, HalfcheetahMorphTasks, HalfcheetahMML

import cv2

os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

In [3]:
wandb.init(
    project='test',
    group='vu',
    name=f'zuxinrui_task',
    mode="online",
)

obstacle_height = np.random.uniform(0.2, 0.5)
obstacle_width = np.random.uniform(0.1, 0.5)
obstacle_spacing = np.random.uniform(0.5, 2.0)
bth_r, bsh_r, bfo_r, fth_r, fsh_r, ffo_r = np.random.uniform(low=0.5, high=2.0, size=6)

env = HalfcheetahMML(
    obstacle_height=0.01,  # (0.2 - 0.5) obstacle_height
    obstacle_width=obstacle_width,  # (0.1 - 0.5)
    obstacle_spacing=obstacle_spacing,  # (0.5 - 2.0)
    n_obstacles=1,  # 10
    design=(bth_r, bsh_r, bfo_r, fth_r, fsh_r, ffo_r),
    backend='spring',
    task_id='backflip',
)

state = jax.jit(env.reset)(rng=jax.random.PRNGKey(seed=0))

url = html.render(env.sys.tree_replace({"opt.timestep": env.dt}), [state.pipeline_state], height=1024)
# with open(os.path.join(exp_dir, f"{exp_name}_{num_steps}.html"), "w") as file:
#     file.write(url)
wandb.log({"env render": wandb.Html(url)})
    # n minibatch really doesn't matter too much!

In [ ]:
episode_length = 150

train_fn = functools.partial(
    ppo_train,
    num_timesteps=10_000_000,
    num_evals=2,
    reward_scaling=1,
    episode_length=episode_length,
    normalize_observations=True,
    action_repeat=1,
    unroll_length=20,
    num_minibatches=32,
    num_updates_per_batch=8,
    discounting=0.95,
    learning_rate=3e-4,
    entropy_cost=0.001,
    num_envs=4096,  # 2048 on 4070 ti s is the fastest  p.s.: num_envs must be divisible by n_batch * batch_size (1+ times per env simulation in the batch)
    batch_size=128,
    seed=3,
)

xdata, ydata = [], []
times = [datetime.now()]

def progress(num_steps, metrics, params, make_policy):
    render(make_policy, params, env, './logs/htmls/', 'halfcheetah', num_steps, metrics)
    times.append(datetime.now())

def render(make_policy, params, env, exp_dir, exp_name, num_steps, metrics=None):
    policy = make_policy(params)
    jit_env_reset = jax.jit(env.reset)
    jit_env_step = jax.jit(env.step)
    jit_policy = jax.jit(policy)

    rollout = []
    key = jax.random.PRNGKey(seed=1)
    key, subkey = jax.random.split(key)
    state = jit_env_reset(rng=subkey)
    for i in range(episode_length):  # 1000 = 50s
        rollout.append(state.pipeline_state)
        key, subkey = jax.random.split(key)
        action, _ = jit_policy(state.obs, subkey)  # Policy requires batched dimension
        # action = action[0]  # Remove batch dimension
        state = jit_env_step(state, action)
        # if i % 1000 == 0:
        #     key, subkey = jax.random.split(key)
        #     state = jit_env_reset(rng=subkey)

    url = html.render(env.sys.tree_replace({"opt.timestep": env.dt}), rollout, height=1024)
    with open(os.path.join(exp_dir, f"{exp_name}_{num_steps}.html"), "w") as file:
        file.write(url)
    wandb.log({
        "video": wandb.Html(url),
        'training/reward': metrics['eval/episode_reward'],
    })

    frames = env.render(rollout, camera='track')
    fps = 30  # Set the desired frames per second of the video
    video_writer = cv2.VideoWriter(
        f'./logs/halfcheetah_{obstacle_height}_{obstacle_width}_{obstacle_spacing}_{env.bth_r}_{env.bsh_r}_{env.bfo_r}_{env.fth_r}_{env.fsh_r}_{env.ffo_r}.mp4', cv2.VideoWriter_fourcc(*"mp4v"), fps, (320, 240)  # remember to switch the x/y axis in cv2
    )
    for i in range(len(frames)):
        video_writer.write(cv2.cvtColor(frames[i], cv2.COLOR_RGB2BGR))
    video_writer.release()

make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)

print(f'time overall: {times[-1] - times[0]}')
print(f'height: {obstacle_height}, width: {obstacle_width}, spacing: {obstacle_spacing}')
print(f'design: {env.bth_r, env.bsh_r, env.bfo_r, env.fth_r, env.fsh_r, env.ffo_r}')

In [12]:
obstacle_height = np.random.uniform(0.2, 0.5)
obstacle_width = np.random.uniform(0.1, 0.5)
obstacle_spacing = np.random.uniform(0.5, 2.0)
bth_r, bsh_r, bfo_r, fth_r, fsh_r, ffo_r = np.random.uniform(low=0.5, high=2.0, size=6)

env = HalfcheetahMML(
    obstacle_height=0.01,  # (0.2 - 0.5) obstacle_height
    obstacle_width=obstacle_width,  # (0.1 - 0.5)
    obstacle_spacing=obstacle_spacing,  # (0.5 - 2.0)
    n_obstacles=1,  # 10
    design=(bth_r, bsh_r, bfo_r, fth_r, fsh_r, ffo_r),
    backend='spring',
    task_id='backflip',
)

state = jax.jit(env.reset)(rng=jax.random.PRNGKey(seed=0))

url = html.render(env.sys.tree_replace({"opt.timestep": env.dt}), [state.pipeline_state], height=1024)
# with open(os.path.join(exp_dir, f"{exp_name}_{num_steps}.html"), "w") as file:
#     file.write(url)
wandb.log({"env render": wandb.Html(url)})

In [21]:
episode_length = 150

train_fn = functools.partial(
    ppo_train,
    num_timesteps=10_000_000,
    num_evals=2,
    reward_scaling=1,
    episode_length=episode_length,
    normalize_observations=True,
    action_repeat=1,
    unroll_length=20,
    num_minibatches=32,
    num_updates_per_batch=8,
    discounting=0.95,
    learning_rate=3e-4,
    entropy_cost=0.001,
    num_envs=4096,  # 2048 on 4070 ti s is the fastest  p.s.: num_envs must be divisible by n_batch * batch_size (1+ times per env simulation in the batch)
    batch_size=128,
    seed=3,
)

xdata, ydata = [], []
times = [datetime.now()]

def progress(num_steps, metrics, params, make_policy):
    render(make_policy, params, env, './logs/htmls/', 'halfcheetah', num_steps, metrics)
    times.append(datetime.now())

def render(make_policy, params, env, exp_dir, exp_name, num_steps, metrics=None):
    policy = make_policy(params)
    jit_env_reset = jax.jit(env.reset)
    jit_env_step = jax.jit(env.step)
    jit_policy = jax.jit(policy)

    rollout = []
    key = jax.random.PRNGKey(seed=1)
    key, subkey = jax.random.split(key)
    state = jit_env_reset(rng=subkey)
    for i in range(episode_length):  # 1000 = 50s
        rollout.append(state.pipeline_state)
        key, subkey = jax.random.split(key)
        action, _ = jit_policy(state.obs, subkey)  # Policy requires batched dimension
        # action = action[0]  # Remove batch dimension
        state = jit_env_step(state, action)
        # if i % 1000 == 0:
        #     key, subkey = jax.random.split(key)
        #     state = jit_env_reset(rng=subkey)

    url = html.render(env.sys.tree_replace({"opt.timestep": env.dt}), rollout, height=1024)
    with open(os.path.join(exp_dir, f"{exp_name}_{num_steps}.html"), "w") as file:
        file.write(url)
    wandb.log({
        "video": wandb.Html(url),
        'training/reward': metrics['eval/episode_reward'],
    })

    frames = env.render(rollout, camera='track')
    fps = 30  # Set the desired frames per second of the video
    video_writer = cv2.VideoWriter(
        f'./logs/halfcheetah_{obstacle_height}_{obstacle_width}_{obstacle_spacing}_{env.bth_r}_{env.bsh_r}_{env.bfo_r}_{env.fth_r}_{env.fsh_r}_{env.ffo_r}.mp4', cv2.VideoWriter_fourcc(*"mp4v"), fps, (320, 240)  # remember to switch the x/y axis in cv2
    )
    for i in range(len(frames)):
        video_writer.write(cv2.cvtColor(frames[i], cv2.COLOR_RGB2BGR))
    video_writer.release()

make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)

print(f'time overall: {times[-1] - times[0]}')
print(f'height: {obstacle_height}, width: {obstacle_width}, spacing: {obstacle_spacing}')
print(f'design: {env.bth_r, env.bsh_r, env.bfo_r, env.fth_r, env.fsh_r, env.ffo_r}')

time overall: 0:00:55.041457
height: 0.4321951640554128, width: 0.28193169626053494, spacing: 1.8636603869625292
design: (1.2354625588988117, 0.7296824273502536, 0.5145062318089433, 0.5294056024553677, 0.82403680896249, 0.9814254598015922)


In [22]:
import cv2


inference_fn = make_inference_fn(params)
jit_env_reset = jax.jit(env.reset)
jit_env_step = jax.jit(env.step)
jit_inference_fn = jax.jit(inference_fn)

rollout = []
rng = jax.random.PRNGKey(seed=1)
state = jit_env_reset(rng=rng)
for _ in range(episode_length):
  rollout.append(state.pipeline_state)
  act_rng, rng = jax.random.split(rng)
  act, _ = jit_inference_fn(state.obs, act_rng)
  state = jit_env_step(state, act)

frames = env.render(rollout, camera='track')
print(frames[0].shape)

# media.show_video(env.render(rollout, camera='side'), fps=1.0 / env.dt)
fps = 30  # Set the desired frames per second of the video
video_writer = cv2.VideoWriter(
    f'./logs/halfcheetah{strftime("%a, %d %b %Y %H:%M:%S +0000", gmtime())}.mp4', cv2.VideoWriter_fourcc(*"mp4v"), fps, (320, 240)  # remember to switch the x/y axis in cv2
)
for i in range(len(frames)):
    video_writer.write(cv2.cvtColor(frames[i], cv2.COLOR_RGB2BGR))
video_writer.release()

(240, 320, 3)


In [4]:
wandb.finish()